<a href="https://colab.research.google.com/github/AdityaManojShinde/AI-Assistant-Chatbot-Powered-by-Groq-API-and-Ollama-for-Local-LLMs/blob/master/Deepfake_Face_Detection_%7C97_94_Test_Accuracy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
xhlulu_140k_real_and_fake_faces_path = kagglehub.dataset_download('xhlulu/140k-real-and-fake-faces', force_download=False)

print('Data source import complete.')


100%|██████████| 3.75G/3.75G [00:17<00:00, 233MB/s]

Extracting files...


Data source import complete.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data.dataloader import DataLoader
from torchvision import datasets, transforms
import torch.optim as optim

In [3]:
# Define transforms (normalize, resize, etc.)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],
                         std = [0.229, 0.224, 0.225])
])

In [4]:
train_data = "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"
test_data = "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test"
valid_data = "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid"

In [5]:
# Load datasets
import os

# Correct the paths using the downloaded dataset's base path
train_data = os.path.join(xhlulu_140k_real_and_fake_faces_path, "real_vs_fake", "real-vs-fake", "train")
test_data = os.path.join(xhlulu_140k_real_and_fake_faces_path, "real_vs_fake", "real-vs-fake", "test")
valid_data = os.path.join(xhlulu_140k_real_and_fake_faces_path, "real_vs_fake", "real-vs-fake", "valid")

train_dataset = datasets.ImageFolder(train_data, transform=transform)
test_dataset  = datasets.ImageFolder(test_data, transform=transform)
valid_dataset = datasets.ImageFolder(valid_data, transform=transform)

In [6]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, prefetch_factor=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=4, prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4, prefetch_factor=2, pin_memory=True)

In [7]:
for image,label in train_loader:
    print(image.shape)
    break

torch.Size([128, 3, 224, 224])


In [8]:
class CNN_Model(nn.Module):
    def __init__(self):
        super(CNN_Model, self).__init__()

        # Define pool ONCE!! ✅
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 1: [128, 3, 224, 224] → [128, 32, 112, 112]
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1, stride=1)
        self.bn1   = nn.BatchNorm2d(32)

        # Block 2: [128, 32, 112, 112] → [128, 64, 56, 56]
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1)
        self.bn2   = nn.BatchNorm2d(64)

        # Block 3: [128, 64, 56, 56] → [128, 128, 28, 28]
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=1)
        self.bn3   = nn.BatchNorm2d(128)

        self.flatten = nn.Flatten()

        # 128 * 28 * 28 = 100352
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.dropout1 = nn.Dropout(0.5)  # prevents overfitting!! ✅

        self.fc2 = nn.Linear(512, 128)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, 2)

    def forward(self, x):
        # Block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # Classifier
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [9]:
# ============================================
# TEST IT!! 👀
# ============================================
model = CNN_Model()
test_input = torch.randn(128, 3, 224, 224)
output = model(test_input)
print(output.shape)
# Should print: torch.Size([128, 2]) ✅

torch.Size([128, 2])


In [10]:
criterion =nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [11]:
import random
import numpy as np
# Fix Python random
random.seed(42)

# Fix numpy random
np.random.seed(42)

# Fix PyTorch random
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [12]:
from tqdm import tqdm

# ✅ Auto detect GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model.to(device)

# ✅ Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=2,
    factor=0.1
)

epochs = 10

for epoch in range(epochs):

    model.train()  # ✅ training mode

    running_loss = 0.0
    total = 0
    correct = 0

    for xb, yb in tqdm(train_loader,
                        desc=f"Epoch {epoch+1}/{epochs}"):
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()  # ✅ first!!
        y_pred = model(xb)
        loss = criterion(y_pred, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = y_pred.max(1)
        total += yb.size(0)
        correct += predicted.eq(yb).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total

    # ✅ scheduler needs a metric
    scheduler.step(train_loss)

    print(f'Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Accuracy: {train_acc:.2f}%')

print("Training completed!!")

Using device: cuda


Epoch 1/10: 100%|██████████| 782/782 [00:39<00:00, 19.87it/s]


Epoch 1/10 | Loss: 0.7988 | Accuracy: 60.08%


Epoch 2/10: 100%|██████████| 782/782 [00:36<00:00, 21.27it/s]


Epoch 2/10 | Loss: 0.5080 | Accuracy: 74.39%


...

Epoch 8/10 | Loss: 0.1529 | Accuracy: 93.92%


Epoch 9/10: 100%|██████████| 782/782 [00:36<00:00, 21.32it/s]


Epoch 9/10 | Loss: 0.1350 | Accuracy: 94.64%


Epoch 10/10: 100%|██████████| 782/782 [00:37<00:00, 21.07it/s]

Epoch 10/10 | Loss: 0.1236 | Accuracy: 95.06%
Training completed!!


In [13]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for xb, yb in test_loader:

        # ✅ move to device!!
        xb, yb = xb.to(device), yb.to(device)

        y_pred = model(xb)

        # ✅ argmax instead of sigmoid!!
        # takes the highest value between 2 outputs
        _, predicted = torch.max(y_pred, 1)

        correct += predicted.eq(yb).sum().item()
        total += yb.size(0)

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')


Test Accuracy: 97.74%


In [14]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for xb, yb in valid_loader:

        # ✅ move to device!!
        xb, yb = xb.to(device), yb.to(device)

        y_pred = model(xb)

        # ✅ argmax instead of sigmoid!!
        # takes the highest value between 2 outputs
        _, predicted = torch.max(y_pred, 1)

        correct += predicted.eq(yb).sum().item()
        total += yb.size(0)

accuracy = correct / total
print(f'validatiton  Accuracy: {accuracy * 100:.2f}%')


validatiton  Accuracy: 97.84%


In [15]:
model.eval()

# Pick one real and one fake from test dataset
real_img, real_label = test_dataset[0]   # check what label this gives
fake_img, fake_label = test_dataset[1]   # check what label this gives


print(f"Dataset label for image 0: {real_label}")
print(f"Dataset label for image 1: {fake_label}")

# Move image to GPU!!
with torch.no_grad():
    real_output = model(real_img.unsqueeze(0).to(device))
    fake_output = model(fake_img.unsqueeze(0).to(device))

    real_pred = torch.argmax(real_output, dim=1).item()
    fake_pred = torch.argmax(fake_output, dim=1).item()

    print(f"Model predicts image 0 as class: {real_pred}")
    print(f"Model predicts image 1 as class: {fake_pred}")

print(f"Classes: {test_dataset.class_to_idx}")

# Check how many real vs fake in test
from collections import Counter
labels = [test_dataset[i][1] for i in range(100)]  # first 100 samples
print(f"Label distribution: {Counter(labels)}")

Dataset label for image 0: 0
Dataset label for image 1: 0
Model predicts image 0 as class: 0
Model predicts image 1 as class: 0
Classes: {'fake': 0, 'real': 1}
Label distribution: Counter({0: 100})


In [16]:
# Check all dataset splits!!
print(f"Train classes: {train_dataset.class_to_idx}")
print(f"Test folder contents:")

import os
for class_name in os.listdir(test_data):
    count = len(os.listdir(os.path.join(test_data, class_name)))
    print(f"  {class_name}: {count} images")


Train classes: {'fake': 0, 'real': 1}
Test folder contents:
  real: 10000 images
  fake: 10000 images


In [17]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        output = model(xb)
        _, predicted = torch.max(output, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(yb.numpy())

print(classification_report(all_labels, all_preds,
      target_names=['fake', 'real']))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


              precision    recall  f1-score   support

        fake       0.98      0.97      0.98     10000
        real       0.97      0.99      0.98     10000

    accuracy                           0.98     20000
   macro avg       0.98      0.98      0.98     20000
weighted avg       0.98      0.98      0.98     20000

Confusion Matrix:
[[9696  304]
 [ 148 9852]]


In [18]:
print(train_dataset.class_to_idx)

{'fake': 0, 'real': 1}


In [19]:
torch.save(model.state_dict(), 'fake_face_detector.pth')

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
import os
import shutil

source_path = '/content/fake_face_detector.pth'
destination_dir = '/content/drive/MyDrive/140k_image_dataset'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Copy the file
shutil.copy(source_path, destination_dir)

print(f"File '{source_path}' saved to '{destination_dir}'")

File '/content/fake_face_detector.pth' saved to '/content/drive/MyDrive/140k_image_dataset'
